In [12]:
import pandas as pd
import re
import os
from pathlib import Path

current_path = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [current_path, *current_path.parents] if (p / "Scripts").is_dir()), None)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the repository.")
# Keep occupations that:
# 1. Require a license Can be self-employed in all states
# 2. Require a bachelor's degree or higher
# 3. Can be self-employed in all states
# 4. Are not split across BLS categories
# 5. Do not include "All Others" in their category

# Paths
base_path = PROJECT_ROOT/"Data/BLS Data/Uniform tables"
output_path = PROJECT_ROOT/"Data/Processed Data/Filtered tables"
professions_path = PROJECT_ROOT/"Data/BLS data/Uniform tables/Professional Licensed Occupations.xlsx"  # Professions path

output_path.mkdir(parents=True, exist_ok=True)
# Professions
df_prof = pd.read_excel(professions_path)  # Professions list


def normalize_title(title):  # Filter text only
    return re.sub(r"[_.\-]", "", str(title).casefold().strip())


# Build the Crosswalk mappings
title_to_orig = {}
code_to_orig = {}

for _, r in df_prof.iterrows(): # Iterate professions
    orig_t = r["Original_Title"]    # Original title
    orig_c = str(r["SOC_Code"]).strip() # Original code

    # Map Titles
    title_to_orig[normalize_title(orig_t)] = (orig_t, orig_c)   # Map normalized original title
    if pd.notna(r.get("New_Title")):    # Map new title
        for t in str(r["New_Title"]).split(";"):
            title_to_orig[normalize_title(t.strip())] = (orig_t, orig_c)

    # Map Codes
    code_to_orig[orig_c] = (orig_t, orig_c) # Map original code
    if pd.notna(r.get("New_SOC_Code")): # Map new code
        for c in str(r["New_SOC_Code"]).split(";"):
            code_to_orig[c.strip()] = (orig_t, orig_c)

for t in range(2005, 2025):
    file_name = f"MSA_{t}_Uniform.xlsx"  # Dataset name
    full_path = os.path.join(base_path, file_name)  # Create path

    if not os.path.exists(full_path):
        continue

    df = pd.read_excel(full_path)   # Read BLS data
    df["OCC_CODE"] = df["OCC_CODE"].astype(str).str.strip() # Clean OCC_CODE

    # Filter by title
    df_filtered_title = df.copy()   # Create a copy
    df_filtered_title["NORM_T"] = df_filtered_title["OCC_TITLE"].apply(normalize_title) # Normalize titles
    df_filtered_title = df_filtered_title[df_filtered_title["NORM_T"].isin(title_to_orig)].copy()   # Map titles

    # Replace with original title and code
    for idx, row in df_filtered_title.iterrows():
        df_filtered_title.at[idx, "OCC_TITLE"], df_filtered_title.at[idx, "OCC_CODE"] = title_to_orig[row["NORM_T"]]

    df_filtered_title = df_filtered_title.drop(columns=["NORM_T"])  # Drop normalized names
    df_filtered_title = df_filtered_title.sort_values(by=list(df_filtered_title.columns)).reset_index(drop=True)    # Sort

    # Filter by code
    df_filtered_code = df.copy()       # Create a copy
    df_filtered_code = df_filtered_code[df_filtered_code["OCC_CODE"].isin(code_to_orig)].copy() # Map codes

    # Additional validation due to the 2021 mess (SOC code recycling)
    df_filtered_code["TEMP_NORM"] = df_filtered_code["OCC_TITLE"].apply(normalize_title)    # Normalize the current titles in the code-filtered dataframe
    df_filtered_code = df_filtered_code[df_filtered_code["TEMP_NORM"].isin(title_to_orig.keys())].copy()    # Only keep rows with matching titles
    df_filtered_code = df_filtered_code.drop(columns=["TEMP_NORM"]) # Drop the temporary column

    # Replace with original title and code
    for idx, row in df_filtered_code.iterrows():
        df_filtered_code.at[idx, "OCC_TITLE"], df_filtered_code.at[idx, "OCC_CODE"] = code_to_orig[row["OCC_CODE"]]

    df_filtered_code = df_filtered_code.sort_values(by=list(df_filtered_code.columns)).reset_index(drop=True)   # Sort

    # Check if identical
    if df_filtered_title.equals(df_filtered_code):
        print(f"{file_name}: Title filter and code filter are identical")
    else:
        print(f"{file_name}: Title filter and code filter are different")

    # Save data
    save_file_name = f"MSA_{t}_Filtered_Extended_Professions.xlsx"
    output_file = os.path.join(output_path, save_file_name)
    df_filtered_title.to_excel(output_file, index=False)

MSA_2005_Uniform.xlsx: Title filter and code filter are identical
MSA_2006_Uniform.xlsx: Title filter and code filter are identical
MSA_2007_Uniform.xlsx: Title filter and code filter are identical
MSA_2008_Uniform.xlsx: Title filter and code filter are identical
MSA_2009_Uniform.xlsx: Title filter and code filter are identical
MSA_2010_Uniform.xlsx: Title filter and code filter are identical
MSA_2011_Uniform.xlsx: Title filter and code filter are identical
MSA_2012_Uniform.xlsx: Title filter and code filter are identical
MSA_2013_Uniform.xlsx: Title filter and code filter are identical
MSA_2014_Uniform.xlsx: Title filter and code filter are identical
MSA_2015_Uniform.xlsx: Title filter and code filter are identical
MSA_2016_Uniform.xlsx: Title filter and code filter are identical
MSA_2017_Uniform.xlsx: Title filter and code filter are identical
MSA_2018_Uniform.xlsx: Title filter and code filter are identical
MSA_2019_Uniform.xlsx: Title filter and code filter are identical
MSA_2020_U